# Gold Layer — Order Summary

### Objective

Create an analytical table with one row per order.

### Grain

One row represents one order.

### Sources

- Silver Orders
- Silver Order Items
- Silver Order Payments

In [0]:
%sql
SELECT *
FROM workspace.silver.olist_orders_dataset
LIMIT 10

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW  order_items_agg AS
SELECT
    order_id,
    COUNT(*) AS n_items,
    COUNT(DISTINCT product_id) AS n_products,
    SUM(price) AS total_item_value,
    SUM(freight_value) AS total_freight
FROM workspace.silver.olist_order_items_dataset
GROUP BY order_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW  payments_agg AS
SELECT
    order_id,
    COUNT(*) AS n_payments,
    SUM(payment_value) AS total_payment
FROM workspace.silver.olist_order_payments_dataset
GROUP BY order_id;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE TABLE workspace.gold.fact_orders
USING DELTA
AS
SELECT
    o.order_id,
    o.customer_id,
    o.purchase_date,
    o.order_status,

    COALESCE(i.n_items, 0) AS n_items,
    COALESCE(i.n_products, 0) AS n_products,
    COALESCE(p.n_payments, 0) AS n_payments,

    COALESCE(i.total_item_value, 0) AS total_item_value,
    COALESCE(i.total_freight, 0) AS total_freight,
    COALESCE(p.total_payment, 0) AS total_payment

FROM workspace.silver.olist_orders_dataset o

LEFT JOIN order_items_agg i
    ON o.order_id = i.order_id

LEFT JOIN payments_agg p
    ON o.order_id = p.order_id;


# Gold Layer — Customer Dimension

### Grain

One row represents one customer order record (`customer_id`).

### Objective

Provide customer attributes for analytical queries.

In [0]:
%sql
CREATE TABLE workspace.gold.dim_customer
USING DELTA
AS
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    customer_city,
    customer_state
FROM workspace.silver.olist_customers_dataset;


# Gold Layer — Date Dimension

### Grain

One row represents one calendar date.

### Objective

Provide reusable calendar attributes for analytical queries.

In [0]:
%sql
SELECT *
FROM workspace.silver.olist_orders_dataset
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT
        MIN(purchase_date) AS start_date,
        MAX(purchase_date) AS end_date
    FROM workspace.silver.olist_orders_dataset
),

dates AS (
    SELECT
        EXPLODE(
            SEQUENCE(
                start_date,
                end_date,
                INTERVAL 1 DAY
            )
        ) AS date
    FROM date_range
)

SELECT
    CAST(date AS DATE) AS date,

    YEAR(date) AS year,
    MONTH(date) AS month,
    QUARTER(date) AS quarter,

    DAY(date) AS day,
    DAYOFWEEK(date) AS day_of_week,

    DATE_FORMAT(date, 'EEEE') AS day_name,
    DATE_FORMAT(date, 'MMMM') AS month_name,

    CASE
        WHEN DAYOFWEEK(date) IN (1, 7)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend

FROM dates;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW brazil_holidays AS

SELECT DATE '2016-01-01' AS holiday_date, 'Confraternização Universal' AS holiday_name
UNION ALL
SELECT DATE '2016-04-21', 'Tiradentes'
UNION ALL
SELECT DATE '2016-05-01', 'Dia do Trabalho'
UNION ALL
SELECT DATE '2016-09-07', 'Independência do Brasil'
UNION ALL
SELECT DATE '2016-10-12', 'Nossa Senhora Aparecida'
UNION ALL
SELECT DATE '2016-11-02', 'Finados'
UNION ALL
SELECT DATE '2016-11-15', 'Proclamação da República'
UNION ALL
SELECT DATE '2016-12-25', 'Natal';

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_date
USING DELTA
AS

WITH holidays AS (

    SELECT DATE '2016-01-01' AS holiday_date, 'Confraternização Universal' AS holiday_name
    UNION ALL SELECT DATE '2016-04-21', 'Tiradentes'
    UNION ALL SELECT DATE '2016-05-01', 'Dia do Trabalho'
    UNION ALL SELECT DATE '2016-09-07', 'Independência do Brasil'
    UNION ALL SELECT DATE '2016-10-12', 'Nossa Senhora Aparecida'
    UNION ALL SELECT DATE '2016-11-02', 'Finados'
    UNION ALL SELECT DATE '2016-11-15', 'Proclamação da República'
    UNION ALL SELECT DATE '2016-12-25', 'Natal'

    -- Replicar para os demais anos do dataset
    -- 2017, 2018, ..., 2018
),

base AS (

    SELECT *
    FROM workspace.gold.dim_date
)

SELECT
    b.*,
    h.holiday_name,
    CASE
        WHEN h.holiday_date IS NOT NULL THEN TRUE
        ELSE FALSE
    END AS is_holiday

FROM base b

LEFT JOIN holidays h
    ON b.date = h.holiday_date;